In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from einops import rearrange
from vqshape.pretrain import LitVQShape
import time
import os

In [ ]:
def process_and_save_representations(data_path, model, output_dir, batch_size=64):
    """
    Process entire dataset in batches and save representations
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Load full dataset
    data = np.load(data_path, allow_pickle=True).item()

    # Process train and test sets separately
    feature_extraction_start = time.time()
    for split in ["train", "test"]:
        X = data[split]["X"]
        y = np.array([int(x) for x in data[split]["y"]])

        token_representations = []
        histogram_representations = []

        # Process in batches
        print(f"Processing {split} set...")
        for i in range(0, len(X), batch_size):
            print(f"Batch {i // batch_size + 1}/{len(X) // batch_size + 1}")

            batch = X[i : i + batch_size]
            x = torch.tensor(batch, dtype=torch.float32).to("cpu")
            x = F.interpolate(x, 256, mode="linear")
            x = rearrange(x, "b c t -> b (c t)")

            with torch.no_grad():
                representations, _ = model(x, mode="tokenize")

            token_repr = representations["token"].cpu().numpy()
            hist_repr = representations["histogram"].cpu().numpy()

            token_representations.append(token_repr)
            histogram_representations.append(hist_repr)

            # Clear memory
            del x, representations
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

        # Concatenate all batches
        token_representations = np.concatenate(token_representations, axis=0)
        histogram_representations = np.concatenate(histogram_representations, axis=0)
        feature_extraction_ends = time.time() - feature_extraction_start
        print(f"Feature extraction time is {feature_extraction_ends} in secs")
        # Save representations
        save_dict = {
            "token_representations": token_representations,
            "histogram_representations": histogram_representations,
            "labels": y,
        }

        output_path = os.path.join(output_dir, f"{split}_representations.npy")
        np.save(output_path, save_dict)
        print(f"Saved {split} representations to {output_path}")

        # Clear memory
        del token_representations, histogram_representations
        torch.cuda.empty_cache() if torch.cuda.is_available() else None


def load_representations(output_dir, split="train"):
    """
    Load saved representations
    """
    path = os.path.join(output_dir, f"{split}_representations.npy")
    data = np.load(path, allow_pickle=True).item()
    return (
        data["token_representations"],
        data["histogram_representations"],
        data["labels"],
    )


# Function to train classifier using saved representations
def train_classifier(output_dir, classifier_type="rf"):
    """
    Train classifier using saved representations
    """
    # Load train data
    train_token, train_hist, train_labels = load_representations(output_dir, "train")

    # Load test data
    test_token, test_hist, test_labels = load_representations(output_dir, "test")

    # Prepare features
    X_train = np.concatenate(
        [train_token.reshape(train_token.shape[0], -1), train_hist], axis=1
    )

    X_test = np.concatenate(
        [test_token.reshape(test_token.shape[0], -1), test_hist], axis=1
    )

    # Train classifier
    if classifier_type == "rf":
        from sklearn.ensemble import RandomForestClassifier

        clf = RandomForestClassifier(n_estimators=100, random_state=42)
    else:
        from sklearn.linear_model import LogisticRegression

        clf = LogisticRegression(max_iter=1000, random_state=42)

    clf.fit(X_train, train_labels)

    # Evaluate
    from sklearn.metrics import accuracy_score, classification_report

    predictions = clf.predict(X_test)
    accuracy = accuracy_score(test_labels, predictions)
    report = classification_report(test_labels, predictions)

    return clf, accuracy, report

In [ ]:
# 1. First, process and save all representations
checkpoint_path = "checkpoints/uea_dim256_codebook512/VQShape.ckpt"  # (download it from original repo, see README)
lit_model = LitVQShape.load_from_checkpoint(checkpoint_path, "cpu")
model = lit_model.model


from pathlib import Path

project_root = Path.cwd().parent.parent
data_path = project_root / "Datasets" / "CMJ.npy"


# Process and save all data
process_and_save_representations(
    data_path=data_path,
    model=model,
    output_dir="./representations",
    batch_size=250,  # Adjust based on your memory constraints
)

# 2. Then, train and evaluate classifier using saved representations
clf, accuracy, report = train_classifier(
    output_dir="./representations",
    classifier_type="rf",  # or 'lr' for logistic regression
)

print(f"Accuracy: {accuracy}")
print("\nClassification Report:")
print(report)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import time


def load_and_prepare_data(output_dir):
    """
    Load saved representations and prepare for classification
    """

    def load_split(split):
        path = f"{output_dir}/{split}_representations.npy"
        data = np.load(path, allow_pickle=True).item()
        # Combine token and histogram representations
        token_flat = data["token_representations"].reshape(
            data["token_representations"].shape[0], -1
        )
        features = np.hstack([token_flat, data["histogram_representations"]])
        return features, data["labels"]

    # Load both splits
    X_train, y_train = load_split("train")
    X_test, y_test = load_split("test")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test


def evaluate_classifiers(X_train, X_test, y_train, y_test):
    """
    Evaluate multiple classifiers and return their performances
    """
    classifiers = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Ridge Classifier": RidgeClassifier(random_state=42),
        "Linear SVM": LinearSVC(random_state=42, max_iter=2000),
    }

    results = {}

    for name, clf in classifiers.items():
        print(f"\nTraining {name}...")
        start_time = time.time()

        # Train
        clf.fit(X_train, y_train)
        training_time = time.time() - start_time

        # Predict
        start_time = time.time()

        y_pred = clf.predict(X_test)

        # Calculate metrics
        train_accuracy = clf.score(X_train, y_train)
        test_accuracy = accuracy_score(y_test, y_pred)
        prediction_time = time.time() - start_time
        conf_matrix = confusion_matrix(y_test, y_pred)
        class_report = classification_report(y_test, y_pred)

        results[name] = {
            "classifier": clf,
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
            "confusion_matrix": conf_matrix,
            "classification_report": class_report,
            "training_time": training_time,
            "prediction_time": prediction_time,
        }

        # Print immediate results
        print(f"\n{name} Results:")
        print(f"Training Accuracy: {train_accuracy:.4f}")
        print(f"Testing Accuracy: {test_accuracy:.4f}")
        print(f"Training Time: {training_time:.2f} seconds")
        print(f"Prediction Time: {prediction_time:.2f} seconds")
        print("\nClassification Report:")
        print(class_report)
        print("\nConfusion Matrix:")
        print(conf_matrix)

    return results


def main_evaluation(output_dir):
    """
    Main function to run the entire evaluation pipeline
    """
    print("Loading and preparing data...")
    X_train, X_test, y_train, y_test = load_and_prepare_data(output_dir)

    print("\nEvaluating classifiers...")
    results = evaluate_classifiers(X_train, X_test, y_train, y_test)

    return results

In [ ]:
# Run the evaluation
results = main_evaluation(output_dir="./representations")

# # Access specific results if needed
lr_results = results["Logistic Regression"]
rf_results = results["Random Forest"]
ridge_results = results["Ridge Classifier"]
svm_results = results["Linear SVM"]

# Print best performing classifier
best_classifier = max(results.items(), key=lambda x: x[1]["test_accuracy"])
print(f"\nBest performing classifier: {best_classifier[0]}")
print(f"Test accuracy: {best_classifier[1]['test_accuracy']:.4f}")